# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 clinical pathology dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library. All dataset entities are referenced by their schema `@id` as per FAIR^2 conventions.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the Croissant schema metadata and review the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nPublished: {metadata.datePublished}")
print(f"Rows, Records expected: Small, n=77\n")

## 2. Data Overview
Review available record sets, fields, and IDs by schema `@id`.

We enumerate all record sets, show their `@id`, and show the first record for each set. As recommended, only `@id` is used for reference.

In [ ]:
# List all RecordSets in the dataset metadata
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} RecordSets:")
for idx, record_set in enumerate(record_sets):
    print(f"  {idx+1}. @id: {record_set['@id']}   name: {record_set.get('name', '')}")
    # Show fields for this RecordSet
    if 'field' in record_set:
        fields = record_set['field'] if isinstance(record_set['field'], list) else [record_set['field']]
        print("      field @id(s):", [f["@id"] for f in fields])
    # Preview the first record (if exists)
    try:
        first_record = next(dataset.records(record_set=record_set['@id']))
        print("      Example record excerpt:", dict(list(first_record.items())[:3]))
    except Exception as e:
        print("      No records preview available.")
    print("\n")

## 3. Data Extraction
Load all data from a specific record set into a DataFrame for analysis.

Below, we identify the record set of main subject-level data (the record set describing cancer survivors and their clinical variables) by its `@id`.

*You may verify the chosen record set `@id` via the overview above for accuracy on future updates.*

In [ ]:
# Replace with actual relevant @id(s) found in the overview step. Here, we illustrate with a likely main RecordSet id found in the schema (adjust as appropriate):

# Get all record set ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Available RecordSet @id's:", record_set_ids)

# We'll select the first (main tabular) RecordSet for analysis
# Update this variable if schema changes or you want a different set
main_record_set_id = record_set_ids[0] if len(record_set_ids) else None

# Load all records
dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print(f"Loaded DataFrame for RecordSet {rid}: shape {df.shape}")

# Show DataFrame info and preview for main dataset
main_df = dataframes[main_record_set_id]
print(f"\nFields in DataFrame for RecordSet @id={main_record_set_id}:")
print(main_df.columns.tolist())
main_df.head(7)

## 4. Exploratory Data Analysis (EDA)

Let's apply common data processing operations to the main DataFrame. All field and column references use their `@id` only.

**Example analysis:**
1. Choose a numeric field (e.g., age, diagnosis interval).
2. Filter for high/low values (removing outliers).
3. Normalize the numeric column.
4. Group by a categorical field (e.g., sex, anatomical_location) and show summary statistics.

*Update @id values below based on schema if a field changes in a future release.*

In [ ]:
# Suggest likely field @ids used for EDA below, UPDATE if schema changes
# For demonstration, use '@id' strings as columns (as loaded by mlcroissant)

print("All fields in main DataFrame:")
for idx, col in enumerate(main_df.columns):
    print(f"  {idx+1}. {col}")

# Example: Age analysis (replace with the actual @id for Age from the schema)
age_field_id = next((c for c in main_df.columns if 'age' in c.lower()), main_df.columns[0])

print(f"\nSelecting numeric field for EDA: {age_field_id}")

# Remove outliers: Ages below 18 or above 100 filtered out
filtered_df = main_df[main_df[age_field_id].apply(lambda x: pd.api.types.is_number(x) and 18 <= float(x) <= 100)]

print(f"Filtered to records with {age_field_id} in [18, 100]. Preview:")
print(filtered_df[[age_field_id]].head())

# Normalize
filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id].astype(float) - filtered_df[age_field_id].astype(float).mean()) / filtered_df[age_field_id].astype(float).std()
print(f"\nNormalized {age_field_id} (z-score):")
print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

# Grouping by a categorical field (e.g., 'sex' or 'anatomical_location')
group_field_id = next((c for c in main_df.columns if 'sex' in c.lower() or 'anatomical' in c.lower()), main_df.columns[0])

if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[age_field_id].agg(['count', 'mean', 'std'])
    print(f"\nGrouped age statistics by {group_field_id}:")
    print(grouped)
else:
    print(f"No suitable group field found (e.g. sex or anatomical_location by @id: {group_field_id})")

## 5. Visualization
Visualize distributions and relationships between two fields—using field `@id`s for labels.

- Distribution plot for the selected numeric field
- Boxplot of the numeric field grouped by chosen categorical field

In [ ]:
# Histogram of the numeric field (e.g. age)
plt.figure(figsize=(7,4))
filtered_df[age_field_id].astype(float).hist(bins=12)
plt.xlabel(age_field_id)
plt.ylabel('Count')
plt.title(f'Distribution of {age_field_id}')
plt.show()

# Boxplot by group field
if group_field_id in filtered_df.columns:\
    plt.figure(figsize=(8,5))
    filtered_df.boxplot(column=age_field_id, by=group_field_id, grid=False)
    plt.title(f'{age_field_id} by {group_field_id}')
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(age_field_id)
    plt.show()
else:
    print(f"No valid group field {group_field_id} found for boxplot.")

## 6. Conclusion

- The FAIR^2 dataset (referenced by Croissant schema and explored via `mlcroissant`) allows programmatic access to all fields, referencing entities via their `@id`.
- Basic EDA shows valid numeric distributions and categorical grouping for clinicopathological variables.
- Process and analyze new fields or add visualizations by selecting the appropriate column `@id` as needed.